In [16]:
import cv2
import numpy as np
import joblib
from skimage import feature

In [17]:
# ----------------- Parámetros y carga de modelos -----------------

# Rutas a los modelos guardados (ajusta si hace falta)
svm_lbp = joblib.load("svm_lbp_barba.joblib")

# Dimensiones usadas al entrenar (deben coincidir con width/height del notebook de entrenamiento)
# Si no las recuerdas, imprime width y height allí y cópialas aquí.
WIDTH =  178 # pon aquí el width que usaste en el entrenamiento
HEIGHT = 218 # pon aquí el height que usaste en el entrenamiento

# Nombres de clases en el orden en que se crearon las carpetas (sorted)
classlabels = ["con_barba", "sin_barba"]  # o ["sin_barba", "con_barba"] según tu orden real

# Clasificador de caras Viola-Jones de OpenCV (incluido en CV2)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

# ----------------- Función LBP coherente con el entrenamiento -----------------

def lbphist(gray, ncellsx, ncellsy, width, height, lbp_method):
    pxpercellx = int(width / ncellsx)
    pxpercelly = int(height / ncellsy)
    ofx = int((width - int(pxpercellx) * ncellsx) / 2)
    ofy = int((height - int(pxpercelly) * ncellsy) / 2)
    LBPu_hist = []
    for i in range(ncellsy):
        for j in range(ncellsx):
            roi = gray[ofy + i * pxpercelly:ofy + (i + 1) * pxpercelly,
                       ofx + j * pxpercellx:ofx + (j + 1) * pxpercellx]
            lbpimg = feature.local_binary_pattern(roi, 8, 1, method=lbp_method)
            n_bins = 256  # igual que en el entrenamiento
            feath, _ = np.histogram(lbpimg, density=False, bins=n_bins, range=(0, n_bins))
            LBPu_hist = np.concatenate([LBPu_hist, feath])
    return LBPu_hist

def get_lbp_descriptor(gray):
    gray_resized = cv2.resize(gray, (WIDTH, HEIGHT))
    feat_lbp = lbphist(gray_resized, ncellsx=3, ncellsy=3, width=WIDTH, height=HEIGHT, lbp_method="nri_uniform")
    return feat_lbp.astype("float32").reshape(1, -1)

# ----------------- Bucle de webcam -----------------

cap = cv2.VideoCapture(0)  # 0 = cámara por defecto

if not cap.isOpened():
    print("No se pudo abrir la cámara.")
else:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Detectar cara
        faces = face_cascade.detectMultiScale(frame_gray, scaleFactor=1.3, minNeighbors=5)

        for (x, y, w, h) in faces:
            # Recorte de cara
            face_gray = frame_gray[y:y+h, x:x+w]

            # Descriptores LBP + predicción
            try:
                desc = get_lbp_descriptor(face_gray)
                pred = svm_lbp.predict(desc)[0]
                label = classlabels[int(pred)]
            except Exception as e:
                label = "error"
                print("Error en predicción:", e)

            # Dibujo sobre el frame
            color = (0, 255, 0) if label == "con_barba" else (0, 0, 255)
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, label, (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

        cv2.imshow("Detector barba / sin barba", frame)

        if cv2.waitKey(1) & 0xFF == 27:  # 27 = ESC
            break

    cap.release()
    cv2.destroyAllWindows()


In [18]:
# ----------------- Parámetros -----------------
# Ajusta estos valores a los que usaste al entrenar
WIDTH = 178   # o el width que imprimiste en tu notebook
HEIGHT = 218  # o el height que imprimiste

classlabels = ["con_barba", "sin_barba"]  # ajusta al orden real

# Rutas a los modelos guardados
svm_lbp = joblib.load("svm_lbp_barba.joblib")

# ----------------- Funciones -----------------

def lbphist(gray, ncellsx, ncellsy, width, height, lbp_method):
    pxpercellx = int(width / ncellsx)
    pxpercelly = int(height / ncellsy)
    ofx = int((width - int(pxpercellx) * ncellsx) / 2)
    ofy = int((height - int(pxpercelly) * ncellsy) / 2)
    LBPu_hist = []
    for i in range(ncellsy):
        for j in range(ncellsx):
            roi = gray[ofy + i * pxpercelly:ofy + (i + 1) * pxpercelly,
                       ofx + j * pxpercellx:ofx + (j + 1) * pxpercellx]
            lbpimg = feature.local_binary_pattern(roi, 8, 1, method=lbp_method)
            n_bins = 256  # igual que en entrenamiento
            feath, _ = np.histogram(lbpimg, density=False, bins=n_bins, range=(0, n_bins))
            LBPu_hist = np.concatenate([LBPu_hist, feath])
    return LBPu_hist

def get_lbp_descriptor_from_image(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray_resized = cv2.resize(gray, (WIDTH, HEIGHT))
    feat_lbp = lbphist(gray_resized, ncellsx=3, ncellsy=3,
                       width=WIDTH, height=HEIGHT, lbp_method="nri_uniform")
    return feat_lbp.astype("float32").reshape(1, -1)

# ----------------- Inferencia sobre una imagen -----------------

# Ruta a una imagen de prueba (cara con o sin barba)
img_path = "dataset_barba/con_barba/000300.jpg"  # pon aquí tu ruta
img = cv2.imread(img_path)
if img is None:
    raise ValueError("No se pudo leer la imagen de prueba")

# Obtener descriptor LBP
desc = get_lbp_descriptor_from_image(img)

# Predicción
pred = svm_lbp.predict(desc)[0]
label = classlabels[int(pred)]

print("Predicción para la imagen:", label)

# (Opcional) Mostrar imagen con etiqueta
cv2.putText(img, label, (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
cv2.imshow("Resultado", img)
cv2.waitKey(0)
cv2.destroyAllWindows()


Predicción para la imagen: sin_barba
